# Spare Calculations


## UCCGSD Cross-Term Complexity

Check the paramter decrease after filtering for cross excitations. The full unrestricted ansatz is constructed below by subclassing QiboChem's `Ansatz_UCCGSD` and disabling `parallel_excitations`, which bypasses `filter_cross_excitations`.

In [2]:
from types import SimpleNamespace

import pandas as pd

from qibochem.ansatz.ucc import Ansatz_UCCGSD
from notebook_utils.general import parse_active_space


class Ansatz_UCCGSDWithCrossTerms(Ansatz_UCCGSD):
    """
    Define the full UCCGSD unrestricted that allows cross terms 
    """
    def excitations(self):
        singles = self._generate_ansatz_excitations(
            rank=1, generalised=True, spin_conserve=True, paired=False,
            spin_adapt=False, parallel_excitations=False,
        )
        doubles = self._generate_ansatz_excitations(
            rank=2, generalised=True, spin_conserve=True, paired=False,
            spin_adapt=False, parallel_excitations=False,
        )
        return {**singles, **doubles}


ACTIVE_SPACES = [
    "2e2o", "2e3o", "4e3o", "4e4o", "4e5o",
    "6e5o", "6e6o", "6e7o", "8e7o", "8e8o",
]

uccgsd_complexity = []
for active_space in ACTIVE_SPACES:
    num_e, num_o = parse_active_space(active_space)
    # Lightweight molecule like object wihtout building the full integrals
    molecule = SimpleNamespace(
        nelec=num_e, n_active_e=num_e,
        nso=2 * num_o, n_active_orbs=2 * num_o,
    )

    full_uccgsd = Ansatz_UCCGSDWithCrossTerms(molecule, use_mp2_guess=False)
    restricted_uccgsd = Ansatz_UCCGSD(molecule, use_mp2_guess=False)
    uccgsd_complexity.append({
        "active_space": active_space,
        "full_parameters": len(full_uccgsd.param_names),
        "full_circuit_depth": full_uccgsd.circuit.depth,
        "restricted_parameters": len(restricted_uccgsd.param_names),
        "restricted_circuit_depth": restricted_uccgsd.circuit.depth,
    })

df_uccgsd_complexity = pd.DataFrame(uccgsd_complexity)
df_uccgsd_complexity["parameter_decrease_percent"] = 100 * (
    1 - df_uccgsd_complexity["restricted_parameters"] / df_uccgsd_complexity["full_parameters"]
)
df_uccgsd_complexity["depth_decrease_percent"] = 100 * (
    1 - df_uccgsd_complexity["restricted_circuit_depth"] / df_uccgsd_complexity["full_circuit_depth"]
)

df_uccgsd_complexity.style.format({
    "full_parameters": "{:,}",
    "full_circuit_depth": "{:,}",
    "restricted_parameters": "{:,}",
    "restricted_circuit_depth": "{:,}",
    "parameter_decrease_percent": "{:.1f}%",
    "depth_decrease_percent": "{:.1f}%",
}).hide(axis="index")


active_space,full_parameters,full_circuit_depth,restricted_parameters,restricted_circuit_depth,parameter_decrease_percent,depth_decrease_percent
2e2o,4,199,3,115,25.0%,42.2%
2e3o,24,"1,775",15,936,37.5%,47.3%
4e3o,24,"1,775",15,936,37.5%,47.3%
4e4o,90,"8,386",52,"4,394",42.2%,47.6%
4e5o,250,"27,364",140,"14,418",44.0%,47.3%
6e5o,250,"27,364",140,"14,418",44.0%,47.3%
6e6o,570,"70,888",315,"37,622",44.7%,46.9%
6e7o,"1,134","157,038",623,"83,827",45.1%,46.6%
8e7o,"1,134","157,038",623,"83,827",45.1%,46.6%
8e8o,"2,044","311,027","1,120","166,795",45.2%,46.4%
